# PaySim Initial Inspection

This notebook performs **inspection only**. It does not train any model and does not convert fraud labels into workload labels.

The current local file is expected to be `PS_20174392719_1491204439457_log.csv`. For exact full-file row/missing/blank counts, prefer the command-line script with `--full-scan`; this notebook intentionally uses a bounded sample for interactive exploration.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

LOCAL_FILE = Path('../data/raw/PS_20174392719_1491204439457_log.csv')
DATASET_ID = 'purulalwani/Synthetic-Financial-Datasets-For-Fraud-Detection'
SPLIT = 'train'
SAMPLE_SIZE = 100_000
RANDOM_SEED = 42

OUTPUT_DIR = Path('../results/paysim_inspection')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Local candidate:', LOCAL_FILE)
print('Configuration ready.')


## Load a bounded exploratory sample

If the local CSV exists, the notebook reads the first `SAMPLE_SIZE` rows. Otherwise it falls back to a streamed Hugging Face sample. This is exploratory only. Use `Research/scripts/inspect_paysim.py --local-file ... --full-scan` for exact full-file row/missing/blank counts.


In [ ]:
if LOCAL_FILE.exists():
    df = pd.read_csv(LOCAL_FILE, nrows=SAMPLE_SIZE)
    source = str(LOCAL_FILE.resolve())
    print('Loaded local bounded sample:', source)
else:
    from datasets import load_dataset
    stream = load_dataset(DATASET_ID, split=SPLIT, streaming=True)
    stream = stream.shuffle(seed=RANDOM_SEED, buffer_size=max(SAMPLE_SIZE * 2, 10_000))
    rows = list(stream.take(SAMPLE_SIZE))
    df = pd.DataFrame(rows)
    source = f'hf://{DATASET_ID}/{SPLIT}'
    print('Local file not found; using streamed sample:', source)

print('Exploratory sample shape:', df.shape)
df.head()


## Columns and data types

In [ ]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
})
schema.to_csv(OUTPUT_DIR / 'schema.csv', index=False)
schema

## Missing values

In [ ]:
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(6),
}).sort_values('missing_count', ascending=False)
missing.to_csv(OUTPUT_DIR / 'missing_values.csv')
missing

## Duplicate rows in the inspected sample

In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_percent = duplicate_count / len(df) * 100 if len(df) else 0
summary = {
    'sample_rows': len(df),
    'duplicate_rows': duplicate_count,
    'duplicate_percent': round(duplicate_percent, 6),
}
(OUTPUT_DIR / 'duplicate_summary.json').write_text(json.dumps(summary, indent=2))
summary

## Transaction-type distribution

In [ ]:
if 'type' in df.columns:
    counts = df['type'].value_counts(dropna=False)
    distribution = pd.DataFrame({
        'count': counts,
        'percent': (counts / len(df) * 100).round(6),
    })
    distribution.to_csv(OUTPUT_DIR / 'transaction_type_distribution.csv')
    display(distribution)
    ax = counts.plot(kind='bar', figsize=(8, 5), title='Transaction Type Distribution')
    ax.set_xlabel('Transaction Type')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'transaction_type_distribution.png', dpi=200)
    plt.show()
else:
    print("'type' column not found.")

## Fraud-label distribution

This is inspected only. It is **not** the latency or workload target.

In [ ]:
if 'isFraud' in df.columns:
    counts = df['isFraud'].value_counts(dropna=False)
    distribution = pd.DataFrame({
        'count': counts,
        'percent': (counts / len(df) * 100).round(8),
    })
    distribution.to_csv(OUTPUT_DIR / 'fraud_distribution.csv')
    display(distribution)
else:
    print("'isFraud' column not found.")

## Amount distribution

In [ ]:
if 'amount' in df.columns:
    amount = pd.to_numeric(df['amount'], errors='coerce')
    summary = amount.describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    summary.to_csv(OUTPUT_DIR / 'amount_summary.csv')
    display(summary)
    upper = amount.quantile(0.99)
    amount.clip(upper=upper).plot(kind='hist', bins=50, figsize=(9, 5), title='Amount Distribution up to P99')
    plt.xlabel('Transaction Amount')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'amount_distribution_p99.png', dpi=200)
    plt.show()
else:
    print("'amount' column not found.")

## Identifier cardinality and leakage candidates

In [ ]:
identifier_rows = []
for column in ['nameOrig', 'nameDest']:
    if column in df.columns:
        unique_count = df[column].nunique(dropna=False)
        identifier_rows.append({
            'column': column,
            'unique_values': unique_count,
            'unique_percent': round(unique_count / len(df) * 100, 6),
        })
identifier_summary = pd.DataFrame(identifier_rows)
identifier_summary.to_csv(OUTPUT_DIR / 'identifier_cardinality.csv', index=False)
display(identifier_summary)

leakage_candidates = [
    c for c in ['isFraud','isFlaggedFraud','newbalanceOrig','newbalanceDest','nameOrig','nameDest']
    if c in df.columns
]
print('Columns requiring leakage review:', leakage_candidates)

## Interpretation reminder

- PaySim provides transaction metadata and fraud-related labels; it does not provide `service_time_ms`.
- `isFraud` is not a Heavy/workload label.
- VPN is not part of the verified current PaySim schema.
- Initial candidate transaction inputs are documented in `../notes/feature_definition_v0.1.md`.
- The next technical step after inspection is controlled repeated execution of the FinCluster reference pipeline to create measured `service_time_ms` ground truth.
